# 02 | Genesis 3D物理環境 × 車輪駆動 × AI異常検知 × 動的巡回
**物理実行の本編**：過去Notebookの2D計画評価とは異なり、**車輪関節の速度指令をGenesis物理エンジンに与え、ロボット剛体の実座標を観測しながら制御します**。
- 3D空間の障害物・設備・差動二輪URDFロボットを自作。
- 遠隔設備観測は**合成センサ値**。Isolation Forestは正常値で学習し、故障正解は意思決定に渡さない。
- A*経路、優先度再計画、制御指令と実位置をログ保存。
- 残存制約：タイヤ摩擦・物理パラメータは簡略化、実機検証なし。

**開始前に00のPNG/MP4スモークテストを成功させてください**。別のColabランタイムを開始して02を実行します。

In [1]:
!apt-get -qq update && apt-get -qq install -y libegl1 libgl1 libgl1-mesa-dri libegl-mesa0 libosmesa6 ffmpeg >/dev/null
%pip -q install 'genesis-world==1.4.4' 'numpy<3' 'scipy<2' 'scikit-learn>=1.3,<2' 'pandas>=2,<3' 'opencv-python-headless>=4.8,<5' 'imageio-ffmpeg'

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.2/408.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.8/286.8 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 

In [2]:
%%writefile pai_patrol.py
"""Physical AI: risk-aware mobile inspection robot (Colab, reproducible demo).

Simulation fidelity: 2-D occupancy grid + differential-drive/unicycle kinematics,
not a ROS/Gazebo or hardware experiment. Remote sensors are synthetic, not measured.
Anomaly ground truth is private to the environment and used only for scoring.
"""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import heapq
import math
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support

STATIONS = {
    'A': (1.5, 7.4),
    'B': (3.5, 6.1),
    'C': (8.6, 7.0),
    'D': (10.5, 3.4),
    'E': (6.5, 1.3),
    'F': (2.0, 3.6),
}
BASE = (1.0, 1.0)
OBSTACLES = [(3.9, 2.1, 5.0, 4.8), (6.5, 4.1, 8.0, 5.7), (8.1, 1.5, 9.0, 3.2)]
WIDTH, HEIGHT = 12.0, 9.0
FEATURES = ['temperature_C', 'vibration_mm_s', 'current_A', 'sound_dB']
STATION_KEYS = list(STATIONS)
MEANS = np.array([[35 + i*1.2, 2.1+i*.08, 5+i*.13, 51+i*.45] for i in range(len(STATIONS))], dtype = float)
STDS = np.array([1.6, .35, .26, 2.1])
FAULT_SHIFT = np.array([9.0, 1.8, 1.2, 9.0])

@dataclass(frozen = True)
class Config:
    dt: float = 0.10
    horizon: float = 150.0
    max_speed: float = 0.68
    max_turn: float = 2.4
    robot_radius: float = 0.25
    grid_res: float = 0.25
    remote_period: float = 2.0
    camera_fps: int = 10
    camera_stride: int = 1
    risk_gate: float = 1.10
    revisit_after: float = 15.0

CFG = Config()

class FloorMap:
    def __init__(self, cfg = CFG):
        self.cfg = cfg
        self.nx = int(round(WIDTH/cfg.grid_res))
        self.ny = int(round(HEIGHT/cfg.grid_res))
        self.grid = np.zeros((self.ny, self.nx), dtype = np.uint8)
        for gy in range(self.ny):
            for gx in range(self.nx):
                x, y = self.from_cell((gx, gy))
                if self.blocked(x, y, margin = cfg.robot_radius+0.09):
                    self.grid[gy, gx] = 1
        assert all(self.grid[self.to_cell(pt)[1], self.to_cell(pt)[0]] == 0 for pt in [BASE, *STATIONS.values()]), 'Station collides with inflated obstacles'
    def blocked(self, x, y, margin = 0.):
        if x<margin or y<margin or x>WIDTH-margin or y>HEIGHT-margin:
            return True
        return any(a-margin <= x <= c+margin and b-margin <= y <= d+margin for a, b, c, d in OBSTACLES)
    def to_cell(self, xy):
        return (min(self.nx-1,max(0,int(xy[0]/self.cfg.grid_res))),
                min(self.ny-1, max(0, int(xy[1]/self.cfg.grid_res))))
    def from_cell(self, cell):
        return ((cell[0]+.5)*self.cfg.grid_res, (cell[1]+.5)*self.cfg.grid_res)
    def line_clear(self, a, b):
        d = math.dist(a, b)
        for u in np.linspace(0, 1, max(2, int(math.ceil(d/0.09)))):
            x = a[0]+(b[0]-a[0])*u
            y = a[1]+(b[1]-a[1])*u
            if self.blocked(x, y, self.cfg.robot_radius+0.04):
                return False
        return True
    def plan(self, start, end):
        s = self.to_cell(start)
        g = self.to_cell(end)
        if s == g or self.line_clear(start, end):
            return [tuple(end)]
        h = lambda p: math.hypot(p[0]-g[0], p[1]-g[1])
        heap = [(h(s), 0., s)]
        best = {s:0.}
        parent = {}
        found = False
        dirs = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if dx or dy]
        while heap:
            _, cost, u = heapq.heappop(heap)
            if cost>best.get(u, float('inf'))+1e-8:
                continue
            if u == g:
                found = True
                break
            for dx, dy in dirs:
                v = (u[0]+dx, u[1]+dy)
                if not (0 <= v[0]<self.nx and 0 <= v[1]<self.ny) or self.grid[v[1], v[0]]:
                    continue
                if dx and dy and (self.grid[u[1], v[0]] or self.grid[v[1], u[0]]):
                    continue
                nc = cost+math.hypot(dx, dy)
                if nc<best.get(v, float('inf')):
                    best[v] = nc
                    parent[v] = u
                    heapq.heappush(heap, (nc+h(v), nc, v))
        if not found:
            raise RuntimeError(f'Route unreachable: {start} -> {end}')
        path = [g]
        while path[-1] != s:
            path.append(parent[path[-1]])
        path.reverse()
        pts = [tuple(start)]+[self.from_cell(pt) for pt in path[1:-1]]+[tuple(end)]
        # Greedy line-of-sight waypoint reduction (collision checked, inflated obstacles)
        out = []
        i = 0
        while i<len(pts)-1:
            j = len(pts)-1
            while j>i+1 and not self.line_clear(pts[i], pts[j]):
                j -= 1
            out.append(pts[j])
            i = j
        return out
    def distance(self, start, end):
        p = [tuple(start)]+self.plan(start, end)
        return sum(math.dist(p[i], p[i+1]) for i in range(len(p)-1))

@dataclass(frozen = True)
class Fault:
    station: str
    onset: float
    severity: float

@dataclass(frozen = True)
class Scenario:
    seed: int
    faults: tuple[Fault, ...]


def make_scenario(seed):
    rng = np.random.default_rng(int(seed))
    keys = rng.choice(STATION_KEYS, size = 2, replace = False)
    return Scenario(int(seed),(Fault(str(keys[0]),0.0,float(rng.uniform(.95,1.45))),
                               Fault(str(keys[1]), float(rng.choice([12., 18., 24.])), float(rng.uniform(1.05, 1.55)))))


def telemetry(scenario, station, t, mode = 'remote', replicate = 0):
    """Synthetic sensor. Policy receives sensor values only, never `scenario.faults`."""
    idx = STATION_KEYS.index(station)
    # Deterministic per (seed, equipment, time-window, mode, replicate).
    tick = max(0, int(t//(CFG.remote_period if mode == 'remote' else 1.0)))
    ss = np.random.SeedSequence([scenario.seed, idx, tick, 0 if mode == 'remote' else 1, int(replicate)])
    rng = np.random.default_rng(ss)
    noise = 1.17 if mode == 'remote' else .40
    x = MEANS[idx]+rng.normal(size = 4)*STDS*noise
    for fault in scenario.faults:
        if station == fault.station and t >= fault.onset:
            x = x+FAULT_SHIFT*fault.severity
    return x

class Detector:
    def __init__(self, seed = 2026):
        rng = np.random.default_rng(seed)
        normal = rng.normal(size = (3500, 4))*1.17
        # Only NORMAL observations in training. Ground truth of evaluation faults unseen.
        self.model = IsolationForest(n_estimators = 120, max_samples = 256, random_state = seed, n_jobs = -1)
        self.model.fit(normal)
        vr = rng.normal(size = (3000, 4))*1.17
        vl = rng.normal(size = (3000, 4))*.40
        rs = -self.model.score_samples(vr)
        ls = -self.model.score_samples(vl)
        self.remote_threshold = float(np.quantile(rs, .99))
        self.local_threshold = float(np.quantile(ls, .995))
        self.reference_75 = float(np.quantile(rs, .75))
        self.spread = max(.015, self.remote_threshold-self.reference_75)
    def features(self, x, station):
        idx = STATION_KEYS.index(station)
        return ((np.asarray(x)-MEANS[idx])/STDS).reshape(1, -1)
    def anomaly(self, x, station):
        return float(-self.model.score_samples(self.features(x, station))[0])
    def remote_risk(self, x, station):
        a = self.anomaly(x, station)
        return float(np.clip((a-self.reference_75)/self.spread, 0., 3.0))
    def onsite_flag(self, scenario, station, t):
        # independent repeat-measurement median reduces occasional false alarms
        scores = [self.anomaly(telemetry(scenario, station, t, mode = 'local', replicate = k), station) for k in range(3)]
        med = float(np.median(scores))
        return med>self.local_threshold, med

@dataclass
class PatrolResult:
    policy: str
    scenario: Scenario
    metrics: dict
    visits: list
    trajectory: list
    events: list
    remote_history: list

class Patrol:
    def __init__(self, scenario, detector, policy = 'adaptive', cfg = CFG, floor = None):
        if policy not in ('adaptive','nearest'):
            raise ValueError('policy must be adaptive or nearest')
        self.scenario = scenario
        self.detector = detector
        self.policy = policy
        self.cfg = cfg
        self.floor = floor if floor is not None else FloorMap(cfg)
        self.pos = np.array(BASE, dtype = float)
        self.heading = math.pi/2
        self.t = 0.
        self.distance = 0.
        self.collisions = 0
        self.replans = 0
        self.route = []
        self.target = None
        self.waypoint = 0
        self.visits = []
        self.events = []
        self.history = []
        self.trajectory = []
        self.last_visit = {k:-1e9 for k in STATION_KEYS}
        self.visit_counts = {k:0 for k in STATION_KEYS}
        self.target_start = -1e9
        self.consecutive_alarm = {k:0 for k in STATION_KEYS}
        self.remote_risks = {k:0. for k in STATION_KEYS}
        self.active_alarm = {k:False for k in STATION_KEYS}
        self.detections = {}
        self.flagged_stations = set()
        self.fp_count = 0
        self.next_remote = 0.
        self.finished = False
    def observe_remote(self):
        risks = {}
        for key in STATION_KEYS:
            observed = telemetry(self.scenario, key, self.t,'remote')
            risk = self.detector.remote_risk(observed, key)
            self.remote_risks[key] = .35*self.remote_risks[key]+.65*risk
            self.consecutive_alarm[key] = (self.consecutive_alarm[key]+1 if self.remote_risks[key] >= self.cfg.risk_gate else 0)
            risks[key] = self.remote_risks[key]
        self.history.append({'t':round(self.t, 1), **risks})
    def options(self):
        outstanding = [k for k in STATION_KEYS if self.last_visit[k]<0]
        if self.policy == 'nearest':
            return outstanding
        alerts=[k for k in STATION_KEYS if self.consecutive_alarm[k]>=2
                and k not in self.flagged_stations and self.visit_counts[k]<2
                and (self.t-self.last_visit[k])>self.cfg.revisit_after]
        return sorted(set(outstanding+alerts))
    def choose_target(self):
        choices = self.options()
        if not choices:
            return 'HOME'
        distances = {k:self.floor.distance(self.pos, STATIONS[k]) for k in choices}
        if self.policy == 'nearest':
            return min(choices, key = lambda k:(distances[k], k))
        # Alarm priority competes with travel cost; no hidden ground truth labels used.
        return max(choices,key=lambda k:(3.5*self.remote_risks[k]-.30*distances[k]
                                         + (1.8 if self.last_visit[k]<0 else 0), -distances[k]))
    def set_target(self, k):
        if self.target == k:
            return
        self.target = k
        self.replans += 1
        self.target_start = self.t
        end = BASE if k == 'HOME' else STATIONS[k]
        self.route = self.floor.plan(self.pos, end)
        self.waypoint = 0
        self.events.append({'t':round(self.t, 1),'kind':'reroute','station':k})
    def arrive(self):
        k = self.target
        if k == 'HOME':
            self.finished = True
            self.events.append({'t':round(self.t, 1),'kind':'home','station':'HOME'})
            return
        flagged, score = self.detector.onsite_flag(self.scenario, k, self.t)
        fault = next((f for f in self.scenario.faults if f.station == k and self.t >= f.onset), None)
        is_fault = fault is not None
        if flagged:
            self.flagged_stations.add(k)
        if flagged and is_fault:
            self.detections.setdefault(k, self.t)
        if flagged and not is_fault:
            self.fp_count += 1
        self.last_visit[k] = self.t
        self.visit_counts[k] += 1
        row={'t':round(self.t,1),'station':k,'flagged':bool(flagged),'fault_active':bool(is_fault),
             'anomaly_score':score,'risk':self.remote_risks[k]}
        self.visits.append(row)
        self.events.append({'t':round(self.t,1),'kind':'inspect','station':k,
                            'flagged':bool(flagged),'fault_active':bool(is_fault)})
        self.target = None
        self.route = []
    def step(self):
        if self.finished:
            return
        if self.t+1e-8 >= self.next_remote:
            self.observe_remote()
            self.next_remote += self.cfg.remote_period
            if self.policy == 'adaptive' and self.target not in (None,'HOME') and self.t-self.target_start>6.:
                # A fresh strong remote alarm can interrupt a low-risk current route.
                opts = self.options()
                serious = [k for k in opts if k != self.target and self.consecutive_alarm[k] >= 2 and self.remote_risks[k]>self.remote_risks.get(self.target, 0)+1.05]
                if serious:
                    best = max(serious, key = lambda k:self.remote_risks[k]-.18*self.floor.distance(self.pos, STATIONS[k]))
                    self.set_target(best)
        if self.target is None:
            self.set_target(self.choose_target())
        goal = np.asarray(BASE if self.target == 'HOME' else STATIONS[self.target])
        if math.dist(self.pos, goal)<.37:
            self.arrive()
        else:
            if self.waypoint >= len(self.route):
                self.route = self.floor.plan(self.pos, goal)
                self.waypoint = 0
                self.replans += 1
            wp = np.asarray(self.route[self.waypoint])
            if np.linalg.norm(wp-self.pos)<.22 and self.waypoint<len(self.route)-1:
                self.waypoint += 1
                wp = np.asarray(self.route[self.waypoint])
            desired = math.atan2(wp[1]-self.pos[1], wp[0]-self.pos[0])
            delta = (desired-self.heading+math.pi)%(2*math.pi)-math.pi
            omega = np.clip(3.0*delta, -self.cfg.max_turn, self.cfg.max_turn)
            # Differential drive kinematics: v=(r/2)(wr+wl), w=(r/L)(wr-wl)
            v = self.cfg.max_speed * max(.05, 1.0-min(1.0, abs(delta)/1.3))
            # Convert commanded twist to actual left/right wheel rates, then integrate.
            wheel_radius, wheel_separation = .085, .36
            wheel_l = (v-.5*wheel_separation*omega)/wheel_radius
            wheel_r = (v+.5*wheel_separation*omega)/wheel_radius
            v_eff = .5*wheel_radius*(wheel_r+wheel_l)
            omega_eff = wheel_radius*(wheel_r-wheel_l)/wheel_separation
            next_heading = (self.heading+omega_eff*self.cfg.dt+math.pi)%(2*math.pi)-math.pi
            proposed = self.pos+v_eff*self.cfg.dt*np.array([math.cos(next_heading), math.sin(next_heading)])
            if self.floor.blocked(*proposed, margin = self.cfg.robot_radius+.01):
                self.collisions += 1
                self.route = self.floor.plan(self.pos, goal)
                self.waypoint = 0
                self.replans += 1
                v = 0.0
            else:
                self.pos = proposed
                self.distance += v*self.cfg.dt
            self.heading = next_heading
        self.t += self.cfg.dt
        self.trajectory.append((self.t,float(self.pos[0]),float(self.pos[1]),float(self.heading),self.target or '-',
                                float(max(self.remote_risks.values())), len(self.detections)))
    def run(self):
        nsteps = int(self.cfg.horizon/self.cfg.dt)
        for _ in range(nsteps):
            if self.finished:
                break
            self.step()
        fault_count = len(self.scenario.faults)
        delays = []
        for f in self.scenario.faults:
            found = self.detections.get(f.station)
            # Missed events receive the horizon penalty (censored observation).
            delays.append((self.cfg.horizon-f.onset if found is None else max(0, found-f.onset), f.severity))
        weight = sum(sev for _, sev in delays)
        weighted = sum(d*sev for d, sev in delays)/weight if weight else float('nan')
        inspected = len([k for k in STATION_KEYS if self.last_visit[k] >= 0])
        metrics={'seed':self.scenario.seed,'policy':self.policy,'detected':len(self.detections),
                 'faults':fault_count,'recall':len(self.detections)/fault_count,
                 'false_alarm_inspections':self.fp_count,
                 'weighted_detection_delay_s':weighted,'distance_m':self.distance,
                 'elapsed_s':self.t,'coverage':inspected/len(STATION_KEYS),
                 'replans':self.replans,'collision_attempts':self.collisions,
                 'home_reached':self.finished}
        return PatrolResult(self.policy, self.scenario, metrics, self.visits, self.trajectory, self.events, self.history)


def evaluate(seed_list = range(30, 50), detector = None, cfg = CFG, verbose = True):
    detector = detector if detector is not None else Detector()
    floor = FloorMap(cfg)
    results = []
    scenarios = []
    for seed in seed_list:
        s = make_scenario(seed)
        scenarios.append(s)
        for policy in ('nearest','adaptive'):
            results.append(Patrol(s, detector, policy, cfg, floor).run())
    dataframe = pd.DataFrame([r.metrics for r in results])
    paired = dataframe.pivot(index = 'seed', columns = 'policy')
    differences=pd.DataFrame({
        'delay_improvement_s':paired[('weighted_detection_delay_s','nearest')]-paired[('weighted_detection_delay_s','adaptive')],
        'recall_improvement':paired[('recall','adaptive')]-paired[('recall','nearest')],
        'extra_distance_m':paired[('distance_m','adaptive')]-paired[('distance_m','nearest')],
    })
    if verbose:
        print('Policies: nearest = distance-only patrol; adaptive = AI-risk priority patrol')
        print('Note: training/calibration and evaluation scenarios are separate.')
        print(dataframe.groupby('policy')[['recall','weighted_detection_delay_s','distance_m','coverage','collision_attempts','false_alarm_inspections']].mean().round(2).to_string())
        print('Paired mean differences (positive delay_improvement is better):')
        print(differences.mean().round(2).to_string())
    return dataframe, differences, results


def smoke_test():
    floor = FloorMap()
    d = Detector()
    for k, p in STATIONS.items():
        assert floor.plan(BASE, p), k
    scenario = make_scenario(42)
    v = Patrol(scenario, d,'adaptive', floor = floor).run()
    assert 0 <= v.metrics['recall'] <= 1
    assert len(v.trajectory)>40
    assert not np.isnan(v.metrics['weighted_detection_delay_s'])
    assert v.metrics['elapsed_s'] <= CFG.horizon+1e-6
    print('smoke test OK:', {k:round(v.metrics[k], 2) if isinstance(v.metrics[k], float) else v.metrics[k] for k in ['recall','distance_m','elapsed_s','coverage','home_reached','collision_attempts']})
    return v

if __name__ == '__main__':
    smoke_test()
    evaluate(range(42, 46))


Writing pai_patrol.py


In [3]:
%%writefile genesis_mission.py
"""Genesis World 3-D rigid-body mobile rover driven by wheel velocity commands.

All observations are synthetic; the simulator is a *simulated* factory. No hardware validation.
The policy receives only synthetic sensor observations, never ground-truth fault IDs/times.
3-D dynamic motion is separately logged; planner-only benchmark is a DIFFERENT experiment.
"""
from __future__ import annotations
import json
import math
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from pai_patrol import (BASE, STATIONS, OBSTACLES, STATION_KEYS, CFG,
                        Detector, Patrol, FloorMap, make_scenario, Scenario)

@dataclass
class MissionSettings:
    dt: float = 0.05
    duration_s: float = 90.0
    fps: int = 10
    backend: str = 'cuda'  # 'cpu' option for diagnostics
    res: tuple[int, int] = (640, 480)
    policy: str = 'adaptive'  # 'nearest' or 'adaptive'
    seed: int = 42
    max_v: float = 0.65
    max_omega: float = 1.9
    wheel_radius: float = 0.12
    wheel_track: float = 0.41
    # URDF positive y wheel axes rotate the rover toward -x for positive omega_y
    wheel_sign: float = -1.0


def generate_rover_urdf(path: Path):
    """Create a simple explicitly inertialized differential drive URDF, no external assets."""
    path = Path(path)
    path.parent.mkdir(parents = True, exist_ok = True)
    text = '''<?xml version="1.0"?>
<robot name="pai_differential_rover">
 <material name="blue"><color rgba="0.13 0.39 0.78 1"/></material>
 <material name="dark"><color rgba="0.12 0.15 0.20 1"/></material>
 <material name="accent"><color rgba="0.95 0.65 0.20 1"/></material>
 <link name="base_link">
   <inertial><origin xyz="0 0 0"/><mass value="6"/><inertia ixx="0.14" ixy="0" ixz="0" iyy="0.18" iyz="0" izz="0.20"/></inertial>
   <visual><geometry><box size="0.50 0.32 0.13"/></geometry><material name="blue"/></visual>
   <collision><geometry><box size="0.50 0.32 0.13"/></geometry></collision>
 </link>
 <link name="left_wheel">
  <inertial><origin xyz="0 0 0"/><mass value="0.5"/><inertia ixx="0.002" ixy="0" ixz="0" iyy="0.002" iyz="0" izz="0.002"/></inertial>
  <visual><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry><material name="dark"/></visual>
  <collision><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry></collision>
 </link>
 <joint name="left_wheel_joint" type="continuous">
  <parent link="base_link"/><child link="left_wheel"/><origin xyz="-0.05 0.205 -0.13" rpy="0 0 0"/>
  <axis xyz="0 1 0"/><limit effort="20" velocity="25"/><dynamics damping="0.03" friction="0.01"/>
 </joint>
 <link name="right_wheel">
  <inertial><origin xyz="0 0 0"/><mass value="0.5"/><inertia ixx="0.002" ixy="0" ixz="0" iyy="0.002" iyz="0" izz="0.002"/></inertial>
  <visual><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry><material name="dark"/></visual>
  <collision><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry></collision>
 </link>
 <joint name="right_wheel_joint" type="continuous">
  <parent link="base_link"/><child link="right_wheel"/><origin xyz="-0.05 -0.205 -0.13" rpy="0 0 0"/>
  <axis xyz="0 1 0"/><limit effort="20" velocity="25"/><dynamics damping="0.03" friction="0.01"/>
 </joint>
 <link name="caster_ball">
  <inertial><origin xyz="0 0 0"/><mass value="0.1"/><inertia ixx="0.00015" ixy="0" ixz="0" iyy="0.00015" iyz="0" izz="0.00015"/></inertial>
  <visual><geometry><sphere radius="0.06"/></geometry><material name="accent"/></visual>
  <collision><geometry><sphere radius="0.06"/></geometry></collision>
 </link>
 <joint name="caster_joint" type="fixed"><parent link="base_link"/><child link="caster_ball"/><origin xyz="0.19 0 -0.19"/></joint>
</robot>'''
    path.write_text(text, encoding = 'utf8')
    return path


def as_numpy(t):
    return t.detach().cpu().numpy() if hasattr(t, 'detach') else np.asarray(t)


def xy_yaw(robot):
    xyz = as_numpy(robot.get_pos()).reshape(-1, 3)[0]
    w, x, y, z = as_numpy(robot.get_quat()).reshape(-1, 4)[0]
    yaw = math.atan2(2*(w*z+x*y), 1-2*(y*y+z*z))
    return (float(xyz[0]), float(xyz[1])), float(yaw), float(xyz[2])


def make_scene(settings:MissionSettings, output:Path):
    import genesis as gs
    if settings.dt <= 0 or settings.duration_s <= 0:
        raise ValueError('Invalid time settings')
    output = Path(output)
    output.mkdir(parents = True, exist_ok = True)
    scene=gs.Scene(show_viewer=False,
                   sim_options = gs.options.SimOptions(dt = settings.dt),
                   vis_options = gs.options.VisOptions(show_world_frame = False))
    scene.add_entity(gs.morphs.Plane(), surface = gs.surfaces.Plastic(color = (.80, .84, .88, 1)))
    # Ground obstacles: rectangles come from same grid as the planner.
    for i, (x1, y1, x2, y2) in enumerate(OBSTACLES):
        scene.add_entity(gs.morphs.Box(pos=((x1+x2)/2,(y1+y2)/2,.45),
                                      size = (x2-x1, y2-y1, .9), fixed = True),
                         surface=gs.surfaces.Plastic(color=(.38,.42,.48,1)), name=f'obstacle_{i}')
    for k, (x, y) in STATIONS.items():
        # Device cabinets are deliberately offset from collision-free inspection points.
        scene.add_entity(gs.morphs.Box(pos=(x+.58,y+.50,.36),
                                      size = (.42, .34, .72), fixed = True),
                         surface=gs.surfaces.Plastic(color=(.16,.50,.45,1)), name=f'equipment_{k}')
        scene.add_entity(gs.morphs.Box(pos=(x,y,.012),size=(.22,.22,.024),fixed=True,collision=False),
                         surface=gs.surfaces.Plastic(color=(.98,.76,.12,1)), name=f'inspect_marker_{k}')
    urdf = generate_rover_urdf(output/'pai_rover.urdf')
    rover=scene.add_entity(gs.morphs.URDF(file=str(urdf.resolve()),
                                           pos = (BASE[0], BASE[1], .25),
                                           fixed = False, merge_fixed_links = True),
                           name = 'pai_rover')
    camera=scene.add_camera(res=settings.res,
                            pos = (15.8, -13.0, 17.8), lookat = (6.0, 4.2, .0), fov = 48, GUI = False)
    scene.build()
    indices = [int(rover.get_joint(n).dofs_idx_local[0]) for n in ('left_wheel_joint','right_wheel_joint')]
    rover.set_dofs_kp(np.zeros(2), indices)
    rover.set_dofs_kv(np.array([7., 7.]), indices)
    rover.set_dofs_force_range(lower = np.array([-18., -18.]), upper = np.array([18., 18.]), dofs_idx_local = indices)
    return scene, rover, camera, indices


def run_3d(settings:MissionSettings, detector:Detector, output:Path, smoke = False):
    """3D wheel-actuated closed-loop run. Returns physical metrics, separate from 2D benchmark."""
    scene, rover, camera, joints = make_scene(settings, output)
    scenario = make_scenario(settings.seed)
    p = Patrol(scenario, detector, policy = settings.policy)
    phys_rows = []
    events = []
    visits = []
    path=Path(output)/('genesis_smoke.mp4' if smoke else f'genesis_{settings.policy}.mp4')
    total_s = 2.0 if smoke else settings.duration_s
    steps = round(total_s/settings.dt)
    last_xy = None
    physical_dist = 0.
    boundaries = 0
    ground_contact_flags = 0
    # Ground truth stays in `scenario`; Patrol only uses it in evaluation and onsite checks.
    camera.start_recording(save_to_filename = str(path.resolve()), fps = settings.fps)
    try:
        for i in range(steps):
            p.t = i*settings.dt
            pos, theta, z = xy_yaw(rover)
            p.pos = np.array(pos)
            p.heading = theta
            if last_xy is not None:
                physical_dist += math.dist(pos, last_xy)
            last_xy = pos
            if p.floor.blocked(*pos, margin = .12):
                boundaries += 1 # geofence/proximity proxy, NOT a measured collision impulse
            if z<-.05:
                ground_contact_flags += 1
            if not smoke:
                if p.t + 1e-6 >= p.next_remote:
                    p.observe_remote()
                    p.next_remote += p.cfg.remote_period
                    if settings.policy == 'adaptive' and p.target not in (None,'HOME') and p.t-p.target_start>6:
                        choices = p.options()
                        serious=[k for k in choices if k!=p.target and
                                p.consecutive_alarm[k] >= 2 and
                                p.remote_risks[k]>p.remote_risks.get(p.target, 0)+1.05]
                        if serious:
                            best = max(serious, key = lambda k:p.remote_risks[k]-.18*p.floor.distance(p.pos, STATIONS[k]))
                            p.set_target(best)
                if p.target is None and not p.finished:
                    p.set_target(p.choose_target())
            else:
                if p.target is None:
                    p.set_target('A')
            v = 0.
            omega = 0.
            if p.target is not None and not p.finished:
                target = np.asarray(BASE if p.target == 'HOME' else STATIONS[p.target])
                if np.linalg.norm(target-p.pos)<.40:
                    if smoke:
                        p.finished = True
                    else:
                        p.arrive()
                if not p.finished and p.target is not None:
                    if p.waypoint >= len(p.route):
                        p.route = p.floor.plan(p.pos, target)
                        p.waypoint = 0
                        p.replans += 1
                    wp = np.asarray(p.route[p.waypoint])
                    if np.linalg.norm(wp-p.pos)<.28 and p.waypoint<len(p.route)-1:
                        p.waypoint += 1
                        wp = np.asarray(p.route[p.waypoint])
                    error = (math.atan2(wp[1]-p.pos[1], wp[0]-p.pos[0])-theta+math.pi)%(2*math.pi)-math.pi
                    omega = float(np.clip(2.2*error, -settings.max_omega, settings.max_omega))
                    v = float(settings.max_v*max(0, 1-abs(error)/.80))
            # Twist -> joint velocities. Sign is determined by URDF wheel joint axes.
            wl = settings.wheel_sign*(v-settings.wheel_track*omega/2)/settings.wheel_radius
            wr = settings.wheel_sign*(v+settings.wheel_track*omega/2)/settings.wheel_radius
            rover.control_dofs_velocity(np.asarray([wl, wr]), dofs_idx_local = joints)
            scene.step()
            phys_rows.append(dict(t=p.t,x=pos[0],y=pos[1],z=z,yaw=theta,
                                  v_command = v, omega_command = omega,
                                  target = p.target or 'NONE',
                                  max_ai_risk = max(p.remote_risks.values()),
                                  completed = p.finished))
            if p.finished and not smoke:
                # Continue recording a stationary robot to preserve a 60-180s video.
                rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        # Recording is also stopped in finally.
    finally:
        try:
            rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        except Exception:
            pass
        camera.stop_recording()
    phys = pd.DataFrame(phys_rows)
    phys.to_csv(Path(output)/'genesis_physical_trace.csv', index = False)
    pd.DataFrame(p.visits).to_csv(Path(output)/'genesis_visits.csv', index = False)
    pd.DataFrame(p.history).to_csv(Path(output)/'genesis_ai_risks.csv', index = False)
    # Note: p.events is the policy control event log; 3D distance derived from rigid body position.
    (Path(output)/'genesis_events.json').write_text(json.dumps(p.events, indent = 2, ensure_ascii = False))
    result={'policy':settings.policy,'seed':settings.seed,'simulated_3d_seconds':total_s,
            'physical_distance_m':physical_dist,'inspected_stations':len(p.visits),
            'replans':p.replans,'geofence_proximity_steps_proxy':boundaries,
            'below_floor_steps':ground_contact_flags,'video':str(path),
            'method':'Genesis wheel-joint velocity control and base pose feedback; synthetic telemetry',
            'warning':'Not a real-world robot experiment; geofence is not a contact sensor.'}
    (Path(output)/'genesis_physical_metrics.json').write_text(json.dumps(result, indent = 2, ensure_ascii = False))
    return result, phys, path


Writing genesis_mission.py


## 環境・AIの初期化

In [4]:
import os
os.environ['PYOPENGL_PLATFORM'] = 'egl'  # fail -> restart runtime, try 'osmesa'
os.environ['PYGLET_HEADLESS'] = 'true'
from pathlib import Path
import genesis as gs
import torch
import pandas as pd
from IPython.display import display, Video
from pai_patrol import Detector
from genesis_mission import MissionSettings, run_3d, generate_rover_urdf
out = Path('outputs')
out.mkdir(exist_ok = True)
gs.init(backend = gs.cuda if torch.cuda.is_available() else gs.cpu, logging_level = 'warning')
print('CUDA available:', torch.cuda.is_available())
detector = Detector(seed = 2026)

CUDA available: True


## 2秒間の車輪制御スモークテスト
API/URDF確認用。**このテスト成功後**に90秒本番を実行してください。

In [5]:
# Genesisで一つのシーンを完成させた後、別のシーンを生成します。
smoke_cfg = MissionSettings(duration_s = 2., fps = 5, res = (480, 320), policy = 'nearest')
smoke_metrics, smoke_trace, smoke_video = run_3d(smoke_cfg, detector, out, smoke = True)
print(smoke_metrics)
assert Path(smoke_video).exists() and len(smoke_trace)>10
print('Movement during smoke:', ((smoke_trace.iloc[-1].x-smoke_trace.iloc[0].x)**2+(smoke_trace.iloc[-1].y-smoke_trace.iloc[0].y)**2)**.5)

/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/call_transformer.py:295: DeprecationWarning: keyword.__init__ got an unexpected keyword argument 'ctx'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  kwarg_node = ast.keyword(
/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


{'policy': 'nearest', 'seed': 42, 'simulated_3d_seconds': 2.0, 'physical_distance_m': 0.17735290025662656, 'inspected_stations': 0, 'replans': 1, 'geofence_proximity_steps_proxy': 0, 'below_floor_steps': 0, 'video': 'outputs/genesis_smoke.mp4', 'method': 'Genesis wheel-joint velocity control and base pose feedback; synthetic telemetry', 'warning': 'Not a real-world robot experiment; geofence is not a contact sensor.'}
Movement during smoke: 0.0377675232394757


## 本番3DシミュレーションとMP4生成
**Colabの描画時間が長くなる可能性があります**。90秒を10fpsで録画。Genesisカメラが自身で録画します。
異常検知の改善指標は01の2D比較であり、ここでは3D物理走行の指標を分離保存します。

In [6]:
cfg = MissionSettings(duration_s = 90.0, fps = 10, res = (640, 480), policy = 'adaptive', seed = 42)
metrics, trace, video_path = run_3d(cfg, detector, out, smoke = False)
print(metrics)
display(trace.tail())
assert 60 <= cfg.duration_s <= 180
assert Path(video_path).exists() and Path(video_path).stat().st_size>1000,'Empty MP4'
if metrics['physical_distance_m']<0.2:
    print('WARNING: rover did not translate sufficiently; see manual (wheel_sign, collisions, gains).')
display(Video(str(video_path), embed = False))

{'policy': 'adaptive', 'seed': 42, 'simulated_3d_seconds': 90.0, 'physical_distance_m': 7.403477848668271, 'inspected_stations': 0, 'replans': 1, 'geofence_proximity_steps_proxy': 0, 'below_floor_steps': 0, 'video': 'outputs/genesis_adaptive.mp4', 'method': 'Genesis wheel-joint velocity control and base pose feedback; synthetic telemetry', 'warning': 'Not a real-world robot experiment; geofence is not a contact sensor.'}


,t,x,y,z,yaw,v_command,omega_command,target,max_ai_risk,completed
1795,89.75,1.138401,1.182420,0.252160,-1.645440,0.0,-1.9,A,2.346987,False
1796,89.80,1.142016,1.190391,0.245004,-1.676716,0.0,-1.9,A,2.346987,False
1797,89.85,1.144889,1.197861,0.246609,-1.645849,0.0,-1.9,A,2.346987,False
1798,89.90,1.148928,1.202669,0.247020,-1.572570,0.0,1.9,A,2.346987,False
1799,89.95,1.150518,1.203888,0.247877,-1.545897,0.0,1.9,A,2.346987,False


## 3D動画に観測可能な状態を重ねる（提出用）
表示する異常リスク・目標設備は3D走行ログから取得します。グラフィックの説明ラベルは英語にして日本語フォント依存を排除します。

In [7]:
import cv2
from pathlib import Path
raw = Path(video_path)
annotated = out/'PAI_final_task_3D.mp4'
reader = cv2.VideoCapture(str(raw))
assert reader.isOpened(),f'Could not open: {raw}'
w = int(reader.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(reader.get(cv2.CAP_PROP_FRAME_HEIGHT))
source_fps = float(reader.get(cv2.CAP_PROP_FPS))
assert source_fps>0 and w>0 and h>0
writer = cv2.VideoWriter(str(annotated), cv2.VideoWriter_fourcc(*'mp4v'), source_fps, (w, h))
assert writer.isOpened(), 'Cannot create output video'
frame_i = 0
try:
    while True:
        ok, frame = reader.read()
        if not ok:
            break
        t = frame_i/source_fps
        idx = min(len(trace)-1, round(t/cfg.dt))
        row = trace.iloc[idx]
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 97), (18, 25, 35), -1)
        cv2.addWeighted(overlay, .77, frame, .23, 0, frame)
        cv2.putText(frame,'PAI 2026 | GENESIS 3D | AUTONOMOUS INSPECTION', (14, 23), cv2.FONT_HERSHEY_SIMPLEX, .46, (250, 250, 250), 1, cv2.LINE_AA)
        cv2.putText(frame,f'Time: {t:05.1f}s | Target: {row.target} | AI risk: {row.max_ai_risk:.2f}',(14,49),cv2.FONT_HERSHEY_SIMPLEX,.58,(0,214,250),2,cv2.LINE_AA)
        cv2.putText(frame,'SYNTHETIC SENSORS - WHEEL-ACTUATED SIMULATION', (14, 78), cv2.FONT_HERSHEY_SIMPLEX, .42, (230, 234, 243), 1, cv2.LINE_AA)
        writer.write(frame)
        frame_i += 1
finally:
    writer.release()
    reader.release()
print('Annotated 3D video:', annotated,'frames:', frame_i,'duration:', frame_i/source_fps)
assert frame_i>0 and 60 <= frame_i/source_fps <= 180
from IPython.display import display, Video
display(Video(str(annotated), embed = False))

Annotated 3D video: outputs/PAI_final_task_3D.mp4 frames: 900 duration: 90.0


## 情報を出力・ダウンロード

In [8]:
import json
import importlib.metadata
import platform
manifest={'python':platform.python_version(),'genesis_world':importlib.metadata.version('genesis-world'),
          'torch':torch.__version__,'cuda':torch.cuda.is_available(),'opengl':os.environ.get('PYOPENGL_PLATFORM'),
          'simulation_settings':cfg.__dict__,'physical_metrics':metrics}
(out/'run_manifest.json').write_text(json.dumps(manifest, indent = 2, ensure_ascii = False, default = str), encoding = 'utf8')
!ls -lh outputs
from google.colab import files
# 以下を必要に応じて個別実行： files.download(str(video_path))

total 2.3M
-rw-r--r-- 1 root root 136K Oct 16 04:04 genesis_adaptive.mp4
-rw-r--r-- 1 root root 5.1K Oct 16 04:04 genesis_ai_risks.csv
-rw-r--r-- 1 root root   67 Oct 16 04:04 genesis_events.json
-rw-r--r-- 1 root root  448 Oct 16 04:04 genesis_physical_metrics.json
-rw-r--r-- 1 root root 215K Oct 16 04:04 genesis_physical_trace.csv
-rw-r--r-- 1 root root 2.4K Oct 16 04:04 genesis_smoke.mp4
-rw-r--r-- 1 root root    1 Oct 16 04:04 genesis_visits.csv
-rw-r--r-- 1 root root 1.9M Oct 16 04:04 PAI_final_task_3D.mp4
-rw-r--r-- 1 root root 2.4K Oct 16 04:04 pai_rover.urdf
-rw-r--r-- 1 root root  919 Oct 16 04:04 run_manifest.json
